# 02 - Value and Policy Networks

## Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Make sure we get outputs from a probabilitistic model each time (for reproducibility)
np.random.seed(1515)

## Same Network, Different Output Head

`01-backpropagation-and-function-approximation.ipynb` built a network with a **Regression Head**: one output number, no activation on the last layer, trained to match a continuous target ($y = x^2$). That network architecture - an input, a hidden layer, an output layer, trained by backpropagation - doesn't change at all between here and the Reinforcement Learning primer that follows this one. What changes is only the **shape and interpretation of the output head**, and what data you train it on.

This notebook covers the two output-head shapes that matter for RL: a **Value Network** (a Regression Head, exactly like notebook 01) and a **Policy Network** (a new shape - a **Classification Head** with a **Softmax** on top).

## Value Networks: Scoring a State

A **Value Network** takes in a description of a state and outputs a single number: how good is this state? This is *architecturally identical* to every regression network already covered in `ml_primer` and notebook 01 - the only thing that's new is what the input vector represents and what the output number means.

For our scouting agent, a state could be a scouted team's stat line - climb rate, auto score, current rank - and the output could be a single number scoring how good an alliance pick that team would be. Let's build that network body (reusing the exact forward-pass shape from notebook 01, just with a 3-number input vector instead of 1) and train it in the ordinary supervised way, on stand-in labels, before RL ever enters the picture.

In [2]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# 3 input features per team: [climb_rate, auto_score, rank_normalized]
# stand-in "pick score" labels for now - a real RL setup replaces these with a reward
# signal, which is exactly what the next primer covers
team_features = np.array([
    [0.90, 4.2, 0.03],   # team 1114: high climb, strong auto, ranked 3rd (normalized)
    [0.10, 2.0, 0.75],   # team 2056: low climb, weak auto, ranked low
    [0.40, 3.0, 0.20],   # team 254: moderate on everything
    [0.95, 1.5, 0.10],   # strong climb, weak auto, ranked well
]).T   # shape (3 features, 4 teams) to match notebook 01's (features, examples) convention

pick_scores = np.array([[9.0, 2.0, 5.5, 6.5]])   # stand-in "how good a pick" label, 0-10

n_features = team_features.shape[0]
W1_v = np.random.randn(4, n_features) * 0.5   # 4 hidden neurons this time
b1_v = np.zeros((4, 1))
W2_v = np.random.randn(1, 4) * 0.5            # Regression Head: 1 output, no activation
b2_v = np.zeros((1, 1))

def value_forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2   # Regression Head: raw output, no squashing activation
    return z1, a1, z2

_, _, values = value_forward(team_features, W1_v, b1_v, W2_v, b2_v)
print(f"Untrained value network outputs: {values.flatten().round(2)}")
print(f"(target pick scores):            {pick_scores.flatten()}")

Untrained value network outputs: [-1.62 -0.43 -1.12 -0.57]
(target pick scores):            [9.  2.  5.5 6.5]


In [3]:
# train with the exact same backprop loop as notebook 01 - only the input width changed
learning_rate = 0.02
n_epochs = 3000
loss_history = []

for epoch in range(n_epochs):
    z1, a1, y_hat = value_forward(team_features, W1_v, b1_v, W2_v, b2_v)
    loss = np.mean((y_hat - pick_scores) ** 2)
    loss_history.append(loss)

    n = team_features.shape[1]
    d_loss_d_yhat = 2 * (y_hat - pick_scores) / n
    d_W2 = d_loss_d_yhat @ a1.T
    d_b2 = np.sum(d_loss_d_yhat, axis=1, keepdims=True)
    d_a1 = W2_v.T @ d_loss_d_yhat
    d_z1 = d_a1 * relu_derivative(z1)
    d_W1 = d_z1 @ team_features.T
    d_b1 = np.sum(d_z1, axis=1, keepdims=True)

    W1_v -= learning_rate * d_W1
    b1_v -= learning_rate * d_b1
    W2_v -= learning_rate * d_W2
    b2_v -= learning_rate * d_b2

_, _, trained_values = value_forward(team_features, W1_v, b1_v, W2_v, b2_v)
print(f"Loss: {loss_history[0]:.2f} -> {loss_history[-1]:.3f}")
print(f"Trained value network outputs: {trained_values.flatten().round(2)}")
print(f"Target pick scores:            {pick_scores.flatten()}")

Loss: 53.10 -> 0.000
Trained value network outputs: [9.  2.  5.5 6.5]
Target pick scores:            [9.  2.  5.5 6.5]


The trained outputs land almost exactly on the stand-in pick scores - in fact the loss goes essentially to zero. That's **overfitting**, not a success: with only 4 teams and 21 weights between the two layers, the network has more than enough capacity to simply memorize 4 numbers rather than learn a real pattern, and there's no held-out validation set here to catch it, unlike every other model in this curriculum (`ml_primer/03`, `cv_primer/03`). That's a deliberate simplification for this notebook, to keep the example small enough to read in one sitting - it is not how you'd actually validate a value network. A real one needs the full train/validation/test discipline from `ml_primer/03-train-tune-eval.ipynb`, and a training set with far more than 4 examples. The part that *does* transfer here is architectural: nothing about the network body was RL-specific. A real value network in an RL setting swaps these hand-picked labels for a reward signal collected from actual outcomes, which is exactly where the next primer picks up.

## Policy Networks: Choosing an Action

A **Policy Network** answers a different question: not "how good is this state," but "which action should I take from this state?" Its output head is a **Classification Head**: one raw output number (a **Logit**) per possible action, converted into a probability distribution with **Softmax**:

$$ \text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}} $$

Every output lands between 0 and 1, and all of them sum to exactly 1 - a valid probability distribution over actions, instead of one single number. This is the same underlying idea as the multi-class classification in `cv_primer/03` (CIFAR-10 had 10 output classes); we're just naming the outputs "actions" instead of "image categories," and looking directly at the probabilities instead of only the final predicted label.

In [4]:
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))   # subtract max for numerical stability
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# same 4 teams, but now 3 possible actions per team: [draft, pass, hold_in_reserve]
W1_p = np.random.randn(4, n_features) * 0.5
b1_p = np.zeros((4, 1))
W2_p = np.random.randn(3, 4) * 0.5   # Classification Head: 3 logits, one per action
b2_p = np.zeros((3, 1))

def policy_forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = relu(z1)
    logits = W2 @ a1 + b2
    action_probs = softmax(logits)
    return action_probs

action_probs = policy_forward(team_features, W1_p, b1_p, W2_p, b2_p)
action_names = ['draft', 'pass', 'hold_in_reserve']

for i in range(team_features.shape[1]):
    print(f"Team {i}: " + ", ".join(
        f"{name}={prob:.2f}" for name, prob in zip(action_names, action_probs[:, i])
    ) + f"  (sums to {action_probs[:, i].sum():.2f})")

Team 0: draft=0.94, pass=0.06, hold_in_reserve=0.00  (sums to 1.00)
Team 1: draft=0.85, pass=0.14, hold_in_reserve=0.01  (sums to 1.00)
Team 2: draft=0.90, pass=0.10, hold_in_reserve=0.00  (sums to 1.00)
Team 3: draft=0.64, pass=0.32, hold_in_reserve=0.04  (sums to 1.00)


Every team's action probabilities sum to 1, exactly as softmax guarantees - this (untrained) policy network is currently just producing close to a uniform guess across the three actions for each team, since nothing has trained it yet. Training a policy network to prefer actions that actually led to good outcomes - rather than labeled examples of the "correct" action, which we don't have here - is a genuinely different training problem from the value network above, and it's the first thing the Reinforcement Learning primer covers.

## The Real Difference Isn't the Architecture

Both networks in this notebook reused the exact same hidden-layer body from notebook 01 - a linear-algebra matrix-vector product, a ReLU, backpropagated with the chain rule. The only architectural difference between a value network, a policy network, and an ordinary supervised regression or classification network is the shape of the output head. What actually makes a network "an RL network" is **what it's trained on**: ordinary supervised learning trains on fixed, pre-labeled correct answers, while Reinforcement Learning trains on a reward signal collected from the consequences of the network's own actions - which is why RL needs its own training loop, covered next, even though the network itself needs nothing new architecturally.

## Try It Yourself

1. Add a 5th team to `team_features` with your own made-up stats (climb rate, auto score, normalized rank) and a stand-in pick score, retrain the value network on all 5 teams, and report its output for your new team.
2. For the *untrained* policy network, verify by hand (using `action_probs`) that increasing one team's `climb_rate` input and re-running `policy_forward` changes that team's action probabilities, without needing any retraining - explain in one sentence why the probabilities change even though no weight has been updated.

In [5]:
# TODO: 1. add a 5th team's features + a stand-in pick score, retrain the value network,
# and print its output for the new team

# TODO: 2. change one team's climb_rate in team_features, re-run policy_forward without
# retraining, and compare the new action_probs to the original


## Resources

- [Sutton & Barto: *Reinforcement Learning: An Introduction*](http://incompleteideas.net/book/the-book.html) - already linked from `ml_primer/00-what-is-ml.md`; Chapter 3 introduces value functions and policies formally, which this notebook previewed informally (textbook).
- [Deep Learning Book: Softmax Units](https://www.deeplearningbook.org/contents/mlp.html) - Goodfellow, Bengio, and Courville's formal treatment of the softmax output layer used above (textbook chapter).
- [3Blue1Brown: Neural Networks](https://www.3blue1brown.com/topics/neural-networks) - revisit the same series linked from `cv_primer/03`, now with the output-head distinction in mind (video).